# Авторегрессия и выборка

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Раскладывать вероятность последовательности на условные вероятности и различать выбор максимума и выборку.

Классификатор текста выдавал вероятности готовых категорий, например положительного и отрицательного отзыва. При генерации распределение выбирает следующий токен из словаря. После выбора вход меняется: к контексту добавляется этот токен, и распределение вычисляется снова. Так повторение знакомого вероятностного выбора строит целую последовательность.

## По одному токену

Авторегрессионная модель описывает последовательность через цепочку $P(x_1,\ldots,x_T)=\prod_{t=1}^T P(x_t\mid x_1,\ldots,x_{t-1})$. При t=1 префикс пуст и первый множитель — просто $P(x_1)$. Каждое следующее распределение зависит от уже доступного префикса. Это правило вероятности, а не утверждение, что модель всегда понимает смысл текста.

Жадный выбор берёт самый вероятный токен на каждом шаге. **Выборка** случайно выбирает токен согласно распределению. Жадный путь не обязан быть самым вероятным целым текстом: лучший первый токен может вести к множеству слабых продолжений. Отдельный токен окончания завершает генерацию; максимальная длина нужна как ограничение бюджета.

## Разобранный пример: вероятность фразы

В начале P(кот)=0.6, P(пёс)=0.4. После «кот» P(спит)=0.5, после «пёс» P(спит)=0.9. Вероятность «кот спит» 0.6·0.5=0.3, «пёс спит» 0.4·0.9=0.36. Самое частое первое слово не дало самую вероятную из этих фраз.

Температура изменяет остроту распределения счётов, top-k оставляет k сильнейших токенов, top-p — набор токенов с достаточной накопленной массой. После ограничения вероятности нужно нормировать заново. Эти правила меняют стиль и разнообразие, но не добавляют фактической достоверности.

## Ограничения демонстрации

В ноутбуке условные вероятности заданы вручную. Генератор с seed повторяет случайную выборку. Это маленькая авторегрессионная таблица, не обученная языковая модель и не симуляция её знаний. Мы отдельно проверяем сумму вероятностей и произведение вдоль пути.

Качество генерации требует вопроса: разнообразие, соответствие условию, истинность или безопасность? Одной вероятности текста недостаточно, чтобы проверить все эти свойства.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

Для полной двухсловной выборки дополним условные распределения: второе слово либо «спит», либо «играет». Шанс «играет» равен 1 минус шанс «спит» после выбранного первого слова. `rng.choice` выбирает один элемент с вероятностями `p`; внутри цикла сначала выбираем существительное, затем зависящий от него глагол. Результат — восемь отдельных фраз, а не одна фраза из восьми независимых первых слов.

In [ ]:
import numpy as np
first = {"кот": 0.6, "пёс": 0.4}
sleep = {"кот": 0.5, "пёс": 0.9}
paths = {word + " спит": first[word] * sleep[word] for word in first}
assert np.isclose(paths["кот спит"], 0.3)
assert np.isclose(paths["пёс спит"], 0.36)
def sample(seed, count=8):
    rng = np.random.default_rng(seed)
    phrases = []
    for _ in range(count):
        noun = str(rng.choice(list(first), p=list(first.values())))
        verb = str(rng.choice(["спит", "играет"], p=[sleep[noun], 1 - sleep[noun]]))
        phrases.append(noun + " " + verb)
    return phrases
assert sample(17) == sample(17)
assert len(sample(17)) == 8
assert set(sample(17)).issubset({"кот спит", "кот играет", "пёс спит", "пёс играет"})
print(paths, sample(17))


## Типичные ошибки

Не называйте вероятный текст истинным. После обрезания распределения нужна нормировка. Задавайте остановку.

## Задачи

### 1. Базовая

Какие токены доступны при авторегрессионном прогнозе?

<details>
<summary>Подсказка</summary>

Только префикс.

</details>



### 2. Базовая

Всегда ли выборка берёт максимум?

<details>
<summary>Подсказка</summary>

Она использует распределение.

</details>



### 3. Прикладная

P(a)=0.5,P(b|a)=0.2. Найдите P(ab).

<details>
<summary>Подсказка</summary>

Перемножьте условные вероятности.

</details>



### 4. Прикладная

Сравните фразы из примера.

<details>
<summary>Подсказка</summary>

Сначала найдите вероятность каждой полной фразы как произведение двух вероятностей, затем сравните.

</details>



### 5. Олимпиадная

Почему жадный выбор не гарантирует самый вероятный полный путь?

<details>
<summary>Подсказка</summary>

Локальная победа не учитывает продолжения.

</details>



### 6. Олимпиадная

Гарантирует ли низкая температура истинность ответа?

<details>
<summary>Подсказка</summary>

Она меняет случайность, а не знания.

</details>



## Итоги и самопроверка

Вычислите вероятность двух путей вручную. Чем воспроизводимость выборки отличается от достоверности результата?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Предыдущие уже заданные или сгенерированные токены.

### Задача 2

Нет. Менее вероятные исходы тоже могут быть выбраны.

### Задача 3

0.1.

### Задача 4

«пёс спит» вероятнее среди этих двух, несмотря на менее частый первый токен.

### Задача 5

Вероятности следующих шагов могут изменить порядок итоговых произведений.

### Задача 6

Нет. Модель может уверенно выбирать ложное продолжение.